In [ ]:
import os, sys, warnings, shutil
import numpy as np
import pandas as pd
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
import sklearn, tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (LSTM as KL, Dense, Dropout,
                                     Input, Concatenate)
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (roc_auc_score, precision_score, recall_score,
                             f1_score, confusion_matrix, roc_curve, auc as skauc)
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.calibration import CalibrationDisplay
from scipy.stats import spearmanr, friedmanchisquare, wilcoxon
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
import shap

warnings.filterwarnings('ignore')

ROOT       = '/kaggle/input/datasets/usamaxai/healthcare/'
MIMIC_HOSP = os.path.join(ROOT, 'MIMIC', 'mimic-iv-3.1', 'hosp')
MIMIC_ICU  = os.path.join(ROOT, 'MIMIC', 'mimic-iv-3.1', 'icu')
EICU       = os.path.join(ROOT, 'EICU', 'eicu-collaborative-research-database-2.0')
OUT        = '/kaggle/working/outputs'
os.makedirs(OUT, exist_ok=True)

N_RUNS  = 30
N_STEPS = 15

VITAL_IDS = {220045:'Heart Rate',
             220181:'Non Invasive Blood Pressure systolic',
             220210:'Respiratory Rate',
             220277:'O2 saturation pulseoxymetry',
             223761:'Temperature Fahrenheit'}
VITAL_NAMES = list(VITAL_IDS.values())

LAB_IDS = {50912:'Creatinine', 51301:'White Blood Cells', 50971:'Potassium',
           50882:'Bicarbonate', 51221:'Hemoglobin', 50931:'Glucose',
           50813:'Lactate'}
VASO_IDS = [221906, 221289, 222315, 221662]
MV_IDS   = [225792, 225794]
SOFA_CHART = {220224:'PaO2', 223835:'FiO2', 220765:'GCS', 220052:'MAP'}
SOFA_LAB   = {51265:'Platelets', 50885:'Bilirubin', 50912:'Creatinine'}
URINE_ID   = 226627

_shap_cache = {}


def log_env():
    import xgboost as xgb, lightgbm as lgb
    env = {'python': sys.version.split()[0], 'numpy': np.__version__,
           'pandas': pd.__version__, 'sklearn': sklearn.__version__,
           'tensorflow': tf.__version__, 'xgboost': xgb.__version__,
           'lightgbm': lgb.__version__, 'shap': shap.__version__,
           'matplotlib': matplotlib.__version__}
    pd.Series(env).to_csv(os.path.join(OUT, 'software_environment.csv'),
                          header=['Version'])
    return env


def compute_sofa(cohort):
    stay_ids = cohort['stay_id'].unique()
    hadm_ids = cohort['hadm_id'].unique()
    cv = []
    for c in pd.read_csv(os.path.join(MIMIC_ICU, 'chartevents.csv'),
                         usecols=['stay_id','itemid','valuenum'],
                         iterator=True, chunksize=8_000_000):
        c = c[c['stay_id'].isin(stay_ids) & c['itemid'].isin(SOFA_CHART)].dropna()
        if not c.empty: cv.append(c)
    if cv:
        cv = pd.concat(cv); cv['f'] = cv['itemid'].map(SOFA_CHART)
        cp = cv.groupby(['stay_id','f'])['valuenum'].mean().unstack()
    else:
        cp = pd.DataFrame()
    lv = []
    for c in pd.read_csv(os.path.join(MIMIC_HOSP, 'labevents.csv'),
                         usecols=['hadm_id','itemid','valuenum'],
                         iterator=True, chunksize=8_000_000):
        c = c[c['hadm_id'].isin(hadm_ids) & c['itemid'].isin(SOFA_LAB)].dropna()
        if not c.empty: lv.append(c)
    if lv:
        lv = pd.concat(lv); lv['f'] = lv['itemid'].map(SOFA_LAB)
        lv = lv.merge(cohort[['hadm_id','stay_id']].drop_duplicates(), on='hadm_id')
        lp = lv.groupby(['stay_id','f'])['valuenum'].mean().unstack()
    else:
        lp = pd.DataFrame()
    inp = pd.read_csv(os.path.join(MIMIC_ICU,'inputevents.csv'),
                      usecols=['stay_id','itemid'])
    vaso = pd.DataFrame({'stay_id': list(set(
        inp[inp['itemid'].isin(VASO_IDS)]['stay_id'])), 'vaso_sf':1})
    try:
        oe = pd.read_csv(os.path.join(MIMIC_ICU,'outputevents.csv'),
                         usecols=['stay_id','itemid','value'])
        urine = (oe[oe['itemid']==URINE_ID].groupby('stay_id')['value']
                 .sum().reset_index().rename(columns={'value':'Urine24h'}))
    except Exception:
        urine = pd.DataFrame(columns=['stay_id','Urine24h'])
    sf = cohort[['stay_id']].drop_duplicates()
    for col in ['PaO2','FiO2','GCS','MAP']:
        sf = sf.merge(cp[[col]], on='stay_id', how='left') \
            if (not cp.empty and col in cp.columns) \
            else sf.assign(**{col: np.nan})
    for col in ['Platelets','Bilirubin','Creatinine']:
        sf = sf.merge(lp[[col]], on='stay_id', how='left') \
            if (not lp.empty and col in lp.columns) \
            else sf.assign(**{col: np.nan})
    sf = sf.merge(vaso, on='stay_id', how='left').merge(urine, on='stay_id', how='left')
    sf['vaso_sf'] = sf['vaso_sf'].fillna(0).astype(int)
    def sr(r):
        if pd.isna(r['PaO2']) or pd.isna(r['FiO2']) or r['FiO2']<=0: return np.nan
        v=r['PaO2']/(r['FiO2']/100.)
        return 0 if v>=400 else 1 if v>=300 else 2 if v>=200 else 3 if v>=100 else 4
    def sc(p): return np.nan if pd.isna(p) else (0 if p>=150 else 1 if p>=100 else 2 if p>=50 else 3 if p>=20 else 4)
    def sl(b): return np.nan if pd.isna(b) else (0 if b<1.2 else 1 if b<=1.9 else 2 if b<=5.9 else 3 if b<=11.9 else 4)
    def sca(r):
        m,v=r['MAP'],r['vaso_sf']
        return np.nan if pd.isna(m) else (3 if v==1 else (0 if m>=70 else 1))
    def scns(g):
        if pd.isna(g): return np.nan
        g=int(round(g))
        return 0 if g==15 else 1 if g>=13 else 2 if g>=10 else 3 if g>=6 else 4
    def sren(r):
        c=r['Creatinine']
        return np.nan if pd.isna(c) else (0 if c<1.2 else 1 if c<=1.9 else 2 if c<=3.4 else 3 if c<=4.9 else 4)
    sf['s1']=sf.apply(sr,axis=1); sf['s2']=sf['Platelets'].apply(sc)
    sf['s3']=sf['Bilirubin'].apply(sl); sf['s4']=sf.apply(sca,axis=1)
    sf['s5']=sf['GCS'].apply(scns); sf['s6']=sf.apply(sren,axis=1)
    comp=['s1','s2','s3','s4','s5','s6']
    for c in comp:
        med=sf[c].median()
        sf[c]=sf[c].fillna(med if not pd.isna(med) else 1)
    sf['SOFA_score']=sf[comp].sum(axis=1)
    return sf[['stay_id','SOFA_score']].copy()


def extract_hourly_vitals(stay_ids_ordered, intime_map):
    sid_set = set(stay_ids_ordered)
    sid_idx = {s:i for i,s in enumerate(stay_ids_ordered)}
    vid_idx = {v:i for i,v in enumerate(VITAL_NAMES)}
    chunks  = []
    for c in pd.read_csv(os.path.join(MIMIC_ICU, 'chartevents.csv'),
                         usecols=['stay_id','itemid','charttime','valuenum'],
                         iterator=True, chunksize=8_000_000):
        c = c[c['stay_id'].isin(sid_set) & c['itemid'].isin(VITAL_IDS)].dropna()
        if not c.empty: chunks.append(c)
    ce = pd.concat(chunks)
    ce['charttime'] = pd.to_datetime(ce['charttime'])
    it = pd.DataFrame({'stay_id': list(intime_map.keys()),
                       'intime':  pd.to_datetime(list(intime_map.values()))})
    ce = ce.merge(it, on='stay_id')
    ce['hour'] = ((ce['charttime']-ce['intime']).dt.total_seconds()/3600).astype(int)
    ce = ce[(ce['hour']>=0)&(ce['hour']<24)]
    ce['vname'] = ce['itemid'].map(VITAL_IDS)
    N = len(stay_ids_ordered)
    X = np.full((N,24,5), np.nan, dtype=np.float32)
    grp = ce.groupby(['stay_id','hour','vname'])['valuenum'].mean()
    for (sid,hr,vn),val in grp.items():
        if sid in sid_idx and vn in vid_idx:
            X[sid_idx[sid], hr, vid_idx[vn]] = val
    for i in range(N):
        for j in range(5):
            s = pd.Series(X[i,:,j]).ffill().bfill()
            X[i,:,j] = s.values
    for j in range(5):
        cm = float(np.nanmean(X[:,:,j]))
        X[:,:,j] = np.where(np.isnan(X[:,:,j]), cm, X[:,:,j])
    means = X.mean(axis=(0,1), keepdims=True)
    stds  = X.std(axis=(0,1),  keepdims=True)
    stds[stds==0] = 1.0
    X_norm = ((X - means) / stds).astype(np.float32)
    agg = np.zeros((N,25), dtype=np.float32)
    for j in range(5):
        agg[:,j*5+0] = X[:,:,j].mean(axis=1)
        agg[:,j*5+1] = X[:,:,j].min(axis=1)
        agg[:,j*5+2] = X[:,:,j].max(axis=1)
        agg[:,j*5+3] = X[:,0,j]
        agg[:,j*5+4] = X[:,23,j]
    return X_norm, agg, means.squeeze(), stds.squeeze()


def preprocess_mimic():
    print('--- MIMIC Preprocessing ---')
    patients  = pd.read_csv(os.path.join(MIMIC_HOSP,'patients.csv'),
                            usecols=['subject_id','gender','anchor_age','dod'])
    admissions= pd.read_csv(os.path.join(MIMIC_HOSP,'admissions.csv'),
                            usecols=['subject_id','hadm_id','race','dischtime'])
    icustays  = pd.read_csv(os.path.join(MIMIC_ICU,'icustays.csv'),
                            usecols=['subject_id','hadm_id','stay_id','intime'])
    diagnoses = pd.read_csv(os.path.join(MIMIC_HOSP,'diagnoses_icd.csv'))
    sep_ids   = diagnoses[diagnoses['icd_code'].isin(
        ['99591','99592','78552'])]['hadm_id'].unique()
    df = (admissions[admissions['hadm_id'].isin(sep_ids)]
          .merge(patients,on='subject_id').merge(icustays,on=['subject_id','hadm_id']))
    df = (df[df['anchor_age']>18].sort_values(['subject_id','intime'])
          .drop_duplicates('subject_id',keep='first'))
    df['mortality_in_hospital']=(
        pd.to_datetime(df['dod']).notna() &
        (pd.to_datetime(df['dod'])<=pd.to_datetime(df['dischtime']))).astype(int)
    def eth(r):
        r=str(r).upper()
        if 'WHITE' in r: return 'White'
        if 'BLACK' in r: return 'Black'
        if 'ASIAN' in r or 'HISPANIC' in r: return 'Asian-Hispanic'
        return 'Other'
    df['ethnicity']  = df['race'].apply(eth)
    df['age_group']  = pd.cut(df['anchor_age'],bins=[18,65,150],
                               labels=['Under65','Over65'],right=False).astype(str)
    df['intersectional_group']=(df['ethnicity']+'_'+df['gender']+'_'+df['age_group'])
    cohort = df[['subject_id','hadm_id','stay_id','gender','anchor_age',
                 'ethnicity','age_group','intersectional_group',
                 'mortality_in_hospital']].copy()
    stay_ids = cohort['stay_id'].unique()
    hadm_ids = cohort['hadm_id'].unique()
    sofa_df  = compute_sofa(cohort)
    vc = [c[c['stay_id'].isin(stay_ids)&c['itemid'].isin(VITAL_IDS)].dropna()
          for c in pd.read_csv(os.path.join(MIMIC_ICU,'chartevents.csv'),
                               usecols=['stay_id','itemid','valuenum'],
                               iterator=True, chunksize=8_000_000)]
    vit = (pd.concat(vc).groupby(['stay_id','itemid'])['valuenum'].mean()
           .reset_index()
           .pivot(index='stay_id', columns='itemid', values='valuenum')
           .rename(columns=VITAL_IDS))
    lc = [c[c['hadm_id'].isin(hadm_ids)&c['itemid'].isin(LAB_IDS)].dropna()
          for c in pd.read_csv(os.path.join(MIMIC_HOSP,'labevents.csv'),
                               usecols=['hadm_id','itemid','valuenum'],
                               iterator=True, chunksize=8_000_000)]
    labs_df = pd.concat(lc)
    labs_ws = labs_df.merge(cohort[['hadm_id','stay_id']].drop_duplicates(),
                             on='hadm_id',how='left').drop_duplicates()
    labs    = (labs_ws.groupby(['stay_id','itemid'])['valuenum'].mean()
               .reset_index()
               .pivot(index='stay_id', columns='itemid', values='valuenum')
               .rename(columns=LAB_IDS))
    diag_agg= (diagnoses.groupby('hadm_id')['icd_code'].nunique()
               .reset_index().rename(columns={'icd_code':'comorbidity_index'}))
    try:
        inp = pd.read_csv(os.path.join(MIMIC_ICU,'inputevents.csv'),
                          usecols=['stay_id','itemid'])
        vp  = pd.DataFrame({'stay_id':list(set(inp[inp['itemid'].isin(VASO_IDS)]['stay_id'])),
                            'vasopressor_use':1})
    except Exception:
        vp  = pd.DataFrame(columns=['stay_id','vasopressor_use'])
    try:
        proc= pd.read_csv(os.path.join(MIMIC_ICU,'procedureevents.csv'),
                          usecols=['stay_id','itemid'])
        mv  = pd.DataFrame({'stay_id':list(set(proc[proc['itemid'].isin(MV_IDS)]['stay_id'])),
                            'mechanical_ventilation':1})
    except Exception:
        mv  = pd.DataFrame(columns=['stay_id','mechanical_ventilation'])
    final = (cohort.merge(vit,on='stay_id',how='left')
             .merge(labs,on='stay_id',how='left')
             .merge(diag_agg,on='hadm_id',how='left')
             .merge(vp,on='stay_id',how='left')
             .merge(mv,on='stay_id',how='left')
             .merge(sofa_df,on='stay_id',how='left'))
    final['vasopressor_use']       =final['vasopressor_use'].fillna(0).astype(int)
    final['mechanical_ventilation']=final['mechanical_ventilation'].fillna(0).astype(int)
    intime_map = (icustays[icustays['stay_id'].isin(stay_ids)]
                  .set_index('stay_id')['intime'].to_dict())
    final = final.drop(columns=['subject_id','hadm_id'])
    num_c = final.select_dtypes(include=np.number).columns.difference(
        ['mortality_in_hospital','stay_id'])
    final[num_c]=SimpleImputer(strategy='median').fit_transform(final[num_c])
    final.to_csv(os.path.join(OUT,'sepsis_cohort_17feat.csv'),index=False)
    pd.DataFrame({'stay_id':list(intime_map.keys()),
                  'intime': list(intime_map.values())}).to_csv(
        os.path.join(OUT,'intime_map.csv'),index=False)
    print(f'Preprocessed: {final.shape}')
    return final, intime_map


def cohort_analysis(df):
    """Create cohort summary tables.

    This version is safe for both the enriched-feature cohort and any
    common-feature / primary regime where SOFA_score, vasopressor_use,
    or mechanical_ventilation may have been intentionally excluded.
    """
    stats = {
        'Total_Patients': len(df),
        'Mean_Age': round(df['anchor_age'].mean(), 2) if 'anchor_age' in df.columns else 'N/A',
        'SD_Age': round(df['anchor_age'].std(), 2) if 'anchor_age' in df.columns else 'N/A',
        'Mortality_Rate': round(df['mortality_in_hospital'].mean(), 4) if 'mortality_in_hospital' in df.columns else 'N/A',
        'Female_n': int((df['gender'] == 'F').sum()) if 'gender' in df.columns else 'N/A',
        'Female_pct': round(100 * (df['gender'] == 'F').mean(), 1) if 'gender' in df.columns else 'N/A',
        'Male_n': int((df['gender'] == 'M').sum()) if 'gender' in df.columns else 'N/A',
        'Male_pct': round(100 * (df['gender'] == 'M').mean(), 1) if 'gender' in df.columns else 'N/A',
        'SOFA_Mean': round(df['SOFA_score'].mean(), 2) if 'SOFA_score' in df.columns else 'N/A',
        'SOFA_SD': round(df['SOFA_score'].std(), 2) if 'SOFA_score' in df.columns else 'N/A',
        'Vasopressor_pct': round(100 * df['vasopressor_use'].mean(), 1) if 'vasopressor_use' in df.columns else 'N/A',
        'MV_pct': round(100 * df['mechanical_ventilation'].mean(), 1) if 'mechanical_ventilation' in df.columns else 'N/A'
    }

    pd.Series(stats).to_csv(os.path.join(OUT, 'table_I_cohort.csv'), header=['Value'])

    required = {'intersectional_group', 'mortality_in_hospital'}
    if required.issubset(df.columns):
        count_col = 'anchor_age' if 'anchor_age' in df.columns else 'mortality_in_hospital'
        (df.groupby('intersectional_group')
         .agg(Count=(count_col, 'size'),
              Prevalence_pct=(count_col, lambda x: round(100 * len(x) / len(df), 2)),
              Mortality_Rate=('mortality_in_hospital', 'mean'))
         .round(3)
         .sort_values('Count', ascending=False)
         .to_csv(os.path.join(OUT, 'table_III_intersectional.csv')))

    return stats


def bootstrap_auc(yt,yp,n=1000,seed=42):
    rng=np.random.RandomState(seed); aucs=[]; N=len(yt)
    for _ in range(n):
        idx=rng.randint(0,N,N); a,b=np.array(yt)[idx],np.array(yp)[idx]
        if len(np.unique(a))<2: continue
        aucs.append(roc_auc_score(a,b))
    return (round(float(np.mean(aucs)),4),
            round(float(np.percentile(aucs,2.5)),4),
            round(float(np.percentile(aucs,97.5)),4))


def fairness_metrics(yt,yp,grp):
    yt=pd.Series(yt).reset_index(drop=True)
    yp=pd.Series(yp).reset_index(drop=True)
    grp=pd.Series(grp).reset_index(drop=True)
    rows=[]
    for g in grp.unique():
        m=grp==g; ytg=yt[m].values; ypg=yp[m].values
        if len(ytg)<2 or len(np.unique(ytg))<2:
            tn,fp,fn,tp=0,0,0,int(ytg.sum())
        else:
            tn,fp,fn,tp=confusion_matrix(ytg,ypg,labels=[0,1]).ravel()
        tpr=tp/(tp+fn) if (tp+fn)>0 else 0.
        fpr=fp/(fp+tn) if (fp+tn)>0 else 0.
        fo =fn/(fn+tn) if (fn+tn)>0 else 0.
        rows.append({'group':g,'TPR':tpr,'FPR':fpr,'FOR':fo,
                     'n':int(m.sum()),'n_pos':int(ytg.sum())})
    dfm=pd.DataFrame(rows).set_index('group')
    gaps={'tpr_gap':float(dfm['TPR'].max()-dfm['TPR'].min()),
          'tpr_std':float(dfm['TPR'].std()),
          'fpr_gap':float(dfm['FPR'].max()-dfm['FPR'].min()),
          'for_gap':float(dfm['FOR'].max()-dfm['FOR'].min())}
    return dfm, gaps


def _prepare_static_matrix(X_static_df, num_cols, cat_cols,
                            num_imp=None, cat_imp=None, ohe=None,
                            scaler=None, fit=False):
    X_num = X_static_df[num_cols].copy() if num_cols else pd.DataFrame(
        index=X_static_df.index)
    X_cat = X_static_df[cat_cols].copy() if cat_cols else pd.DataFrame(
        index=X_static_df.index)

    if fit:
        num_imp = SimpleImputer(strategy='median')
        if num_cols:
            X_num_a = num_imp.fit_transform(X_num)
        else:
            X_num_a = np.zeros((len(X_static_df), 0))
        cat_imp = SimpleImputer(strategy='most_frequent')
        if cat_cols:
            X_cat_a = cat_imp.fit_transform(X_cat.astype(str))
            ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
            X_cat_a = ohe.fit_transform(X_cat_a)
        else:
            X_cat_a = np.zeros((len(X_static_df), 0))
            ohe = None
        X_full = np.hstack([X_num_a, X_cat_a]).astype(np.float32)
        scaler = StandardScaler().fit(X_full)
        X_out  = scaler.transform(X_full).astype(np.float32)
    else:
        if num_cols:
            X_num_a = num_imp.transform(X_num)
        else:
            X_num_a = np.zeros((len(X_static_df), 0))
        if cat_cols and ohe is not None:
            X_cat_a = cat_imp.transform(X_cat.astype(str))
            X_cat_a = ohe.transform(X_cat_a)
        else:
            X_cat_a = np.zeros((len(X_static_df), 0))
        X_full = np.hstack([X_num_a, X_cat_a]).astype(np.float32)
        X_out  = scaler.transform(X_full).astype(np.float32)

    return X_out, {'num_imp':num_imp,'cat_imp':cat_imp,
                   'ohe':ohe,'scaler':scaler}


def ml_baselines(df, intime_map):
    print('\n--- ML Baselines ---')
    drop = ['mortality_in_hospital','ethnicity','age_group',
             'intersectional_group','stay_id']
    static_cols = [c for c in df.columns if c not in drop+VITAL_NAMES]
    y = df['mortality_in_hospital'].values

    idx = np.arange(len(df))
    tr_i, te_i = train_test_split(idx, test_size=0.2,
                                   random_state=42, stratify=y)

    Xst_tr_df = df.iloc[tr_i][static_cols]
    Xst_te_df = df.iloc[te_i][static_cols]
    y_tr = y[tr_i]; y_te = y[te_i]

    num_cols = Xst_tr_df.select_dtypes(include=np.number).columns.tolist()
    cat_cols = Xst_tr_df.select_dtypes(
        include=['object','category']).columns.tolist()

    sid_ordered = df['stay_id'].tolist()
    X_seq, X_agg, vit_means, vit_stds = extract_hourly_vitals(
        sid_ordered, intime_map)

    Xst_tr_n, st_obj = _prepare_static_matrix(
        Xst_tr_df, num_cols, cat_cols, fit=True)
    Xst_te_n, _ = _prepare_static_matrix(
        Xst_te_df, num_cols, cat_cols,
        num_imp=st_obj['num_imp'], cat_imp=st_obj['cat_imp'],
        ohe=st_obj['ohe'], scaler=st_obj['scaler'], fit=False)

    n_static = Xst_tr_n.shape[1]
    X_seq_tr = X_seq[tr_i]; X_seq_te = X_seq[te_i]

    t_in   = Input(shape=(24,5), name='temporal')
    x      = KL(64, return_sequences=True, activation='tanh')(t_in)
    x      = Dropout(0.3)(x)
    x      = KL(32, activation='tanh')(x)
    x      = Dropout(0.3)(x)
    s_in   = Input(shape=(n_static,), name='static')
    merged = Concatenate()([x, s_in])
    merged = Dense(16, activation='relu')(merged)
    out    = Dense(1, activation='sigmoid')(merged)
    lstm   = Model(inputs=[t_in, s_in], outputs=out)
    lstm.compile(optimizer='adam', loss='binary_crossentropy')
    lstm.fit([X_seq_tr, Xst_tr_n], y_tr, epochs=20, batch_size=128,
             validation_split=0.1, verbose=0,
             callbacks=[EarlyStopping(monitor='val_loss', patience=3,
                                      restore_best_weights=True)])
    lstm_pb = lstm.predict([X_seq_te, Xst_te_n], verbose=0).ravel()
    lstm_pr = (lstm_pb>0.5).astype(int)
    lstm_auc, ci_lo, ci_hi = bootstrap_auc(y_te, lstm_pb)
    pd.DataFrame({'AUC':[lstm_auc],'CI_lo':[ci_lo],'CI_hi':[ci_hi]}).to_csv(
        os.path.join(OUT,'lstm_ci.csv'), index=False)
    print(f'LSTM AUC={lstm_auc} (95% CI: {ci_lo}-{ci_hi})')

    X_tab_tr = np.hstack([X_agg[tr_i], Xst_tr_n])
    X_tab_te = np.hstack([X_agg[te_i], Xst_te_n])

    spos = float((y_tr==0).sum()/max((y_tr==1).sum(),1))
    ml_mods = {
        'Logistic Regression': LogisticRegression(max_iter=1000,
                                class_weight='balanced', random_state=42),
        'Random Forest':       RandomForestClassifier(n_estimators=200,
                                class_weight='balanced', random_state=42, n_jobs=-1),
        'XGBoost':             XGBClassifier(n_estimators=200,
                                eval_metric='logloss', scale_pos_weight=spos,
                                random_state=42, n_jobs=-1),
        'LightGBM':            LGBMClassifier(n_estimators=200,
                                class_weight='balanced', random_state=42,
                                n_jobs=-1, verbose=-1)}
    rows = []
    fig, ax = plt.subplots(figsize=(9,7))
    for name, est in ml_mods.items():
        est.fit(X_tab_tr, y_tr)
        pr = est.predict(X_tab_te)
        pb = est.predict_proba(X_tab_te)[:,1]
        fpr_c, tpr_c, _ = roc_curve(y_te, pb)
        av = skauc(fpr_c, tpr_c)
        ax.plot(fpr_c, tpr_c, label=f'{name} (AUC={av:.3f})')
        rows.append({'Model':name,'AUC':round(av,4),
                     'Precision':round(precision_score(y_te,pr,zero_division=0),4),
                     'Recall':round(recall_score(y_te,pr),4),
                     'F1':round(f1_score(y_te,pr),4)})

    if 'SOFA_score' in static_cols:
        cb_tr = Xst_tr_df[['SOFA_score']].fillna(
            Xst_tr_df['SOFA_score'].median()).values
        cb_te = Xst_te_df[['SOFA_score']].fillna(
            Xst_tr_df['SOFA_score'].median()).values
        cb_nm = 'SOFA-Only (Clinical Baseline)'
    else:
        cb_tr = Xst_tr_df[['anchor_age']].fillna(0).values
        cb_te = Xst_te_df[['anchor_age']].fillna(0).values
        cb_nm = 'Age-Only (Clinical Baseline)'
    cb_lr = LogisticRegression(max_iter=1000).fit(cb_tr, y_tr)
    cb_pb = cb_lr.predict_proba(cb_te)[:,1]
    cb_pr = cb_lr.predict(cb_te)
    fpr_c, tpr_c, _ = roc_curve(y_te, cb_pb)
    cb_auc = skauc(fpr_c, tpr_c)
    ax.plot(fpr_c, tpr_c, '--', label=f'{cb_nm} (AUC={cb_auc:.3f})')
    rows.append({'Model':cb_nm,'AUC':round(cb_auc,4),
                 'Precision':round(precision_score(y_te,cb_pr,zero_division=0),4),
                 'Recall':round(recall_score(y_te,cb_pr),4),
                 'F1':round(f1_score(y_te,cb_pr),4)})

    fpr_c, tpr_c, _ = roc_curve(y_te, lstm_pb)
    ax.plot(fpr_c, tpr_c,
            label=f'LSTM (AUC={lstm_auc:.3f}, 95% CI: {ci_lo}-{ci_hi})')
    rows.append({'Model':'LSTM','AUC':lstm_auc,
                 'Precision':round(precision_score(y_te,lstm_pr,zero_division=0),4),
                 'Recall':round(recall_score(y_te,lstm_pr),4),
                 'F1':round(f1_score(y_te,lstm_pr),4)})
    ax.plot([0,1],[0,1],'k--',alpha=0.4,label='Random Classifier')
    ax.set_xlabel('False Positive Rate')
    ax.set_ylabel('True Positive Rate')
    ax.legend(loc='lower right', fontsize=8)
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(OUT,'figure_2_roc.png'), dpi=300)
    plt.close()
    tbl = pd.DataFrame(rows).set_index('Model')
    tbl.to_csv(os.path.join(OUT,'table_II_baselines.csv'))
    print(tbl.to_string())
    return (lstm, st_obj, X_seq, vit_means, vit_stds, static_cols,
            num_cols, cat_cols,
            {'auc':lstm_auc,'ci_lo':ci_lo,'ci_hi':ci_hi})


def static_audit(models, X_te, y_te, g_te):
    yt = y_te.reset_index(drop=True)
    gt = g_te.reset_index(drop=True)
    rows = []
    for name, model in models.items():
        pr = model.predict(X_te)
        pb = model.predict_proba(X_te)[:,1]
        dfm, gaps = fairness_metrics(yt, pd.Series(pr), gt)
        rows.append({'Model':name,
                     'AUC':round(roc_auc_score(yt,pb),4),
                     'Precision':round(precision_score(yt,pr,zero_division=0),4),
                     'Recall':round(recall_score(yt,pr),4),
                     'F1':round(f1_score(yt,pr),4),
                     'TPR_Gap':round(gaps['tpr_gap'],4),
                     'sigma_TPR':round(gaps['tpr_std'],4),
                     'FPR_Gap':round(gaps['fpr_gap'],4),
                     'FOR_Gap':round(gaps['for_gap'],4)})
        fig,(ax1,ax2)=plt.subplots(2,1,figsize=(8,10),
                                    gridspec_kw={'height_ratios':[3,1]})
        ax1.plot([0,1],[0,1],'k:')
        CalibrationDisplay.from_estimator(model,X_te,yt,n_bins=10,
                                           name=name,ax=ax1)
        ax2.hist(pb,range=(0,1),bins=10)
        ax2.set_xlabel('Predicted probability')
        fig.tight_layout()
        fig.savefig(os.path.join(OUT,
            f'figure_3_cal_{name[:4].lower()}.png'), dpi=300)
        plt.close()
    audit = pd.DataFrame(rows).set_index('Model')
    audit.to_csv(os.path.join(OUT,'table_IV_static_audit.csv'))
    print(audit.to_string())
    return audit


def run_shap(model, X_tr, step):
    pp = model.named_steps['pp']
    clf = model.named_steps['clf']
    Xp = pp.fit_transform(X_tr)
    fns = pp.get_feature_names_out()
    expl = shap.TreeExplainer(clf)
    sv = expl(Xp)
    vals = sv.values
    if vals.ndim==3: vals = vals[:,:,1]
    _shap_cache[step] = pd.Series(np.abs(vals).mean(axis=0), index=fns)
    label = 'initial' if step==0 else 'final'
    plt.figure()
    shap.summary_plot(vals, pd.DataFrame(Xp, columns=fns), show=False)
    plt.title(f'SHAP Feature Importance (t={step})')
    plt.tight_layout()
    plt.savefig(os.path.join(OUT,f'figure_shap_{label}_t{step}.png'), dpi=300)
    plt.close()


def shap_spearman():
    steps = sorted(_shap_cache.keys())
    if len(steps)<2: return {}
    t0,tf = _shap_cache[steps[0]], _shap_cache[steps[-1]]
    common = t0.index.intersection(tf.index)
    rho,pv = spearmanr(t0[common].values, tf[common].values)
    res = {'Spearman_rho':round(rho,4),'p_value':round(pv,6)}
    pd.DataFrame({'Metric':list(res.keys()),
                  'Value':list(res.values())}).to_csv(
        os.path.join(OUT,'shap_spearman.csv'), index=False)
    print(f'SHAP rho={rho:.4f} p={pv:.6f}')
    return res


# === MAIN SIMULATION FUNCTION with DAF learning rate and normalization flag ===
def run_one_sim(df, run_seed, capture_shap=False, static_weight=1.5, gamma=0.005,
                daf_lr=0.1, normalize_daf=True):
    T0=0.9; TF=0.2; ALPHA=0.025; BETA=0.01; GAMMA=gamma;
    model_names = ['Baseline','Static FAR','Dynamic DAF']
    hist = {m:{'auc':[],'tpr_gap':[],'tpr_std':[],'fpr_gap':[],'for_gap':[]}
            for m in model_names}
    th=[]; dh=[]
    # separate feedback pool per strategy
    pools = {m: df.copy() for m in model_names}
    trust = T0
    groups = sorted(df['intersectional_group'].unique())
    daf_w = {g:1.0 for g in groups}
    tpr_g = {g:[] for g in groups}
    majority = df['intersectional_group'].value_counts().idxmax()
    if run_seed == 42:
        print(f'  Static FAR majority anchor: {majority}')

    drop = ['mortality_in_hospital','ethnicity','age_group',
            'intersectional_group','stay_id']
    pc = [c for c in df.columns if c not in drop]
    nf = df[pc].select_dtypes(include=np.number).columns.tolist()
    cf = df[pc].select_dtypes(include=['object','category']).columns.tolist()

    for step in range(N_STEPS):
        dh.append(daf_w.copy())
        last_daf = {}
        models_step = {}
        for name in model_names:
            pool = pools[name]
            tr, te = train_test_split(pool, test_size=0.3,
                                       random_state=run_seed+step,
                                       stratify=pool['mortality_in_hospital'])
            ytr, yte = tr['mortality_in_hospital'], te['mortality_in_hospital']
            gtr, gte = tr['intersectional_group'],  te['intersectional_group']
            Xtr, Xte = tr[pc], te[pc]
            pp = ColumnTransformer([
                ('num', StandardScaler(), nf),
                ('cat', OneHotEncoder(handle_unknown='ignore',
                                      sparse_output=False), cf)
            ], remainder='drop')
            if name == 'Baseline':
                w = None
            elif name == 'Static FAR':
                w = gtr.apply(lambda g: 1.0 if g == majority else static_weight).values
            else:  # Dynamic DAF
                w = gtr.map(daf_w).fillna(1.0).values
            model = Pipeline([('pp',pp),
                              ('clf',RandomForestClassifier(
                                  random_state=run_seed,
                                  class_weight='balanced', n_jobs=-1))])
            if w is None:
                model.fit(Xtr, ytr)
            else:
                model.fit(Xtr, ytr, clf__sample_weight=w)
            models_step[name] = (model, Xte, yte, gte, Xtr)

            pr = model.predict(Xte)
            pb = model.predict_proba(Xte)[:,1]
            hist[name]['auc'].append(roc_auc_score(yte,pb))
            dfm, gaps = fairness_metrics(
                yte.reset_index(drop=True),
                pd.Series(pr),
                gte.reset_index(drop=True))
            for k in ['tpr_gap','tpr_std','fpr_gap','for_gap']:
                hist[name][k].append(gaps[k])
            if name == 'Dynamic DAF':
                last_daf = dfm['TPR'].to_dict()
            if name == 'Baseline':
                for g in groups:
                    tpr_g[g].append(float(dfm['TPR'].get(g, 0.)))

        if capture_shap and step in [0, N_STEPS-1]:
            mb, _, _, _, Xtr_b = models_step['Baseline']
            run_shap(mb, Xtr_b.copy(), step)

        # DAF update with optional normalization
        if last_daf:
            ref = np.mean(list(last_daf.values()))
            for g in groups:
                gap = ref - last_daf.get(g, ref)
                daf_w[g] *= np.exp(daf_lr * gap)
            if normalize_daf:
                mean_w = np.mean(list(daf_w.values()))
                if mean_w > 0:
                    for g in groups:
                        daf_w[g] /= mean_w

        # feedback perturbation
        for name in model_names:
            model, Xte, yte, gte, _ = models_step[name]
            pool = pools[name]
            pr_b = model.predict(Xte)
            fp = int(((pr_b==1)&(yte==0)).sum())
            tp = int(((pr_b==1)&(yte==1)).sum())
            if name == 'Baseline':
                trust += BETA*tp - ALPHA*fp - GAMMA*(fp+tp)/max(len(yte),1)
                trust = max(TF, min(1.0, trust))
                th.append(trust)
            low = pool[model.predict(pool[pc])==0].index
            miss = low[np.random.rand(len(low)) < (1-trust)*0.5]
            if len(miss):
                for c in ['Heart Rate','Respiratory Rate']:
                    if c in pool.columns:
                        pool.loc[miss,c] *= (1+np.random.uniform(0.02,0.05,len(miss)))
                for c in ['Non Invasive Blood Pressure systolic',
                          'O2 saturation pulseoxymetry']:
                    if c in pool.columns:
                        pool.loc[miss,c] *= (1-np.random.uniform(0.02,0.05,len(miss)))
                if 'Lactate' in pool.columns:
                    pool.loc[miss,'Lactate'] *= (1+np.random.uniform(0.05,0.15,len(miss)))

    return ({m:{'final_tpr_gap':hist[m]['tpr_gap'][-1],
                'final_tpr_std':hist[m]['tpr_std'][-1],
                'final_fpr_gap':hist[m]['fpr_gap'][-1],
                'final_for_gap':hist[m]['for_gap'][-1],
                'final_auc':hist[m]['auc'][-1],
                **{k+'_series':hist[m][k] for k in
                   ['tpr_gap','tpr_std','fpr_gap','for_gap','auc']}}
             for m in hist}, th, tpr_g, dh)


# Helper functions (unchanged)
def _trajectory_metrics(series):
    s = np.asarray(series, dtype=float)
    if len(s) < 2:
        return {'Mean_Abs_Cycle_Change':0.0,
                'Max_Cycle_Jump':0.0,
                'Trajectory_Range':0.0}
    d = np.abs(np.diff(s))
    return {'Mean_Abs_Cycle_Change':float(np.mean(d)),
            'Max_Cycle_Jump':float(np.max(d)),
            'Trajectory_Range':float(np.max(s)-np.min(s))}


def _paired_stats(arrays_by_name, out_csv):
    names = list(arrays_by_name.keys())
    rows = []
    if len(names) >= 3:
        vals = [arrays_by_name[n] for n in names]
        try:
            chi2, p = friedmanchisquare(*vals)
            rows.append({'Metric':'Friedman','Comparison':'All',
                         'Statistic':round(float(chi2),4),
                         'p':round(float(p),6)})
        except Exception:
            rows.append({'Metric':'Friedman','Comparison':'All',
                         'Statistic':'N/A','p':'N/A'})
    for i in range(len(names)):
        for j in range(i+1, len(names)):
            a = np.asarray(arrays_by_name[names[i]], dtype=float)
            b = np.asarray(arrays_by_name[names[j]], dtype=float)
            if len(a) != len(b) or np.allclose(a-b, 0):
                stat, p = np.nan, np.nan
            else:
                stat, p = wilcoxon(a, b)
            rows.append({'Metric':'Wilcoxon',
                         'Comparison':f'{names[i]} vs {names[j]}',
                         'Statistic':round(float(stat),4) if not np.isnan(stat) else 'N/A',
                         'p':round(float(p),6) if not np.isnan(p) else 'N/A'})
    pd.DataFrame(rows).to_csv(os.path.join(OUT, out_csv), index=False)
    return rows


def trajectory_stability_analysis(series_agg, model_names):
    metric_names = ['Mean_Abs_Cycle_Change','Max_Cycle_Jump','Trajectory_Range']
    vals = {metric:{m:[] for m in model_names} for metric in metric_names}
    for m in model_names:
        for s in series_agg[m]['tpr_gap']:
            met = _trajectory_metrics(s)
            for metric in metric_names:
                vals[metric][m].append(met[metric])
    rows = []
    for m in model_names:
        row = {'Model_Strategy':m}
        for metric in metric_names:
            arr = np.asarray(vals[metric][m], dtype=float)
            row[f'{metric}_mean'] = round(float(np.mean(arr)),4)
            row[f'{metric}_SD'] = round(float(np.std(arr)),4)
        rows.append(row)
    summary = pd.DataFrame(rows).set_index('Model_Strategy')
    summary.to_csv(os.path.join(OUT,'table_trajectory_stability.csv'))
    stat_rows = []
    for metric in metric_names:
        stats = _paired_stats(vals[metric], f'stats_trajectory_{metric}.csv')
        for r in stats:
            r = dict(r)
            r['Trajectory_Metric'] = metric
            stat_rows.append(r)
    pd.DataFrame(stat_rows).to_csv(os.path.join(OUT,'stats_trajectory_stability.csv'), index=False)
    try:
        fig, ax = plt.subplots(figsize=(9,6))
        ax.boxplot([vals['Mean_Abs_Cycle_Change'][m] for m in model_names],
                   labels=model_names, patch_artist=True,
                   boxprops=dict(facecolor='lightblue'),
                   medianprops=dict(color='red', linewidth=2))
        ax.set_ylabel('Mean Absolute Cycle-to-Cycle TPR-Gap Change')
        ax.grid(alpha=0.3)
        plt.tight_layout()
        plt.savefig(os.path.join(OUT,'figure_trajectory_stability_boxplot.png'), dpi=300)
        plt.close()
    except Exception:
        pass
    print('\nTrajectory stability:\n', summary.to_string())
    return summary, stat_rows


def run_simulation(df, daf_lr=0.1, normalize_daf=True):
    print(f'\n--- Simulation ({N_RUNS}x{N_STEPS}) ---')
    mn = ['Baseline','Static FAR','Dynamic DAF']
    agg = {m:{'tpr_gap':[],'tpr_std':[],'fpr_gap':[],'for_gap':[],'auc':[]}
           for m in mn}
    series_agg = {m:{'tpr_gap':[],'tpr_std':[],'fpr_gap':[],'for_gap':[],'auc':[]}
                  for m in mn}
    r0h=r0t=r0g=r0d=None
    for run in range(N_RUNS):
        seed = 42 + run*100
        np.random.seed(seed)
        rs, th, tg, dh = run_one_sim(df, seed, capture_shap=(run==0),
                                     daf_lr=daf_lr, normalize_daf=normalize_daf)
        for m in mn:
            for k in ['tpr_gap','tpr_std','fpr_gap','for_gap','auc']:
                agg[m][k].append(rs[m][f'final_{k}'])
                series_agg[m][k].append(rs[m][f'{k}_series'])
        if run==0: r0h,r0t,r0g,r0d = rs,th,tg,dh
        if (run+1)%5==0: print(f'  {run+1}/{N_RUNS}')

    ta = [agg[m]['tpr_gap'] for m in mn]
    fs, fp = friedmanchisquare(*ta)
    bonf = 0.05/3
    wr = []
    for m1, m2 in [('Baseline','Static FAR'),
                    ('Baseline','Dynamic DAF'),
                    ('Static FAR','Dynamic DAF')]:
        d = np.array(agg[m1]['tpr_gap']) - np.array(agg[m2]['tpr_gap'])
        if np.all(d==0): ws,wp = np.nan, np.nan
        else: ws,wp = wilcoxon(agg[m1]['tpr_gap'], agg[m2]['tpr_gap'])
        wr.append({'Comparison':f'{m1} vs {m2}',
                   'W':round(ws,3) if not np.isnan(ws) else 'N/A',
                   'p':round(wp,4) if not np.isnan(wp) else 'N/A',
                   'Sig_Bonf':(wp<bonf) if not np.isnan(wp) else False})
    fa = [agg[m]['fpr_gap'] for m in mn]
    ffs, ffp = friedmanchisquare(*fa)
    fwr = []
    for m1, m2 in [('Baseline','Static FAR'),
                    ('Baseline','Dynamic DAF'),
                    ('Static FAR','Dynamic DAF')]:
        d = np.array(agg[m1]['fpr_gap']) - np.array(agg[m2]['fpr_gap'])
        if np.all(d==0): ws,wp = np.nan, np.nan
        else: ws,wp = wilcoxon(agg[m1]['fpr_gap'], agg[m2]['fpr_gap'])
        fwr.append({'Comparison':f'{m1} vs {m2}',
                    'W':round(ws,3) if not np.isnan(ws) else 'N/A',
                    'p':round(wp,4) if not np.isnan(wp) else 'N/A'})
    pd.DataFrame([{'Test':'Friedman_TPR','chi2':round(fs,3),
                   'p':round(fp,4)},
                  {'Test':'Friedman_FPR','chi2':round(ffs,3),
                   'p':round(ffp,4)}]).to_csv(
        os.path.join(OUT,'stats_friedman.csv'), index=False)
    pd.DataFrame(wr).to_csv(os.path.join(OUT,'stats_wilcoxon_tpr.csv'),
                              index=False)
    pd.DataFrame(fwr).to_csv(os.path.join(OUT,'stats_wilcoxon_fpr.csv'),
                               index=False)
    print(f'TPR Friedman chi2={fs:.3f} p={fp:.4f}')
    print(f'FPR Friedman chi2={ffs:.3f} p={ffp:.4f}')
    for r in wr: print(f'  {r["Comparison"]}: p={r["p"]}')
    srows = []
    for m in mn:
        d = agg[m]
        srows.append({'Model_Strategy':m,
            'Final_TPR_Gap_mean':round(np.mean(d['tpr_gap']),4),
            'Final_TPR_Gap_SD':  round(np.std(d['tpr_gap']),4),
            'sigma_TPR_mean':    round(np.mean(d['tpr_std']),4),
            'sigma_TPR_SD':      round(np.std(d['tpr_std']),4),
            'FPR_Gap_mean':      round(np.mean(d['fpr_gap']),4),
            'FPR_Gap_SD':        round(np.std(d['fpr_gap']),4),
            'FOR_Gap_mean':      round(np.mean(d['for_gap']),4),
            'FOR_Gap_SD':        round(np.std(d['for_gap']),4),
            'AUC_mean':          round(np.mean(d['auc']),4),
            'AUC_SD':            round(np.std(d['auc']),4)})
    summ = pd.DataFrame(srows).set_index('Model_Strategy')
    summ.to_csv(os.path.join(OUT,'table_VI_simulation.csv'))
    print('\nTable VI:\n', summ.to_string())
    pd.DataFrame(r0d[-1], index=['Final_Weight']).T.round(4).to_csv(
        os.path.join(OUT,'table_V_daf_weights.csv'))
    pd.DataFrame(r0g).round(4).to_csv(
        os.path.join(OUT,'tpr_by_group.csv'))
    traj_summary, traj_stats = trajectory_stability_analysis(series_agg, mn)
    make_figures(r0h, r0t, r0g, r0d, agg)
    return summ, {'friedman_chi2':round(fs,3),'friedman_p':round(fp,4),
                  'wilcoxon':wr,'fpr_friedman_chi2':round(ffs,3),
                  'fpr_friedman_p':round(ffp,4),
                  'trajectory_summary':traj_summary,
                  'trajectory_stats':traj_stats}


def make_figures(h0, trust, tg, dh, agg):
    st = {'Baseline':{'marker':'o','linestyle':'--','color':'steelblue'},
          'Static FAR':{'marker':'s','linestyle':':','color':'darkorange'},
          'Dynamic DAF':{'marker':'^','linestyle':'-','color':'green'}}
    def sv(fn):
        plt.tight_layout()
        plt.savefig(os.path.join(OUT,fn), dpi=300)
        plt.close()
    steps = range(N_STEPS)
    for key,ylabel,fname in [
        ('tpr_gap_series','Max-Min TPR Gap','figure_5_tpr_gap.png'),
        ('tpr_std_series','Std Dev of Subgroup TPRs','figure_6_tpr_std.png'),
        ('fpr_gap_series','Max-Min FPR Gap','figure_8_fpr_gap.png'),
        ('for_gap_series','Max-Min FOR Gap','figure_9_for_gap.png')]:
        plt.figure(figsize=(10,6))
        for n,s in st.items():
            plt.plot(steps, h0[n][key], **s, label=n)
        plt.xlabel('Simulation Time Step (t)')
        plt.ylabel(ylabel); plt.legend(); plt.grid(alpha=0.3)
        sv(fname)
    plt.figure(figsize=(10,6))
    for n,s in st.items():
        plt.plot(steps, h0[n]['auc_series'], **s, label=n)
    plt.xlabel('Simulation Time Step (t)')
    plt.ylabel('AUC-ROC'); plt.legend(); plt.grid(alpha=0.3)
    sv('figure_7_auc.png')
    plt.figure(figsize=(10,6))
    plt.plot(steps, trust, 'd-', color='purple')
    plt.xlabel('Simulation Time Step (t)')
    plt.ylabel('Trust Score (T_t)')
    plt.ylim(0,1.05); plt.grid(alpha=0.3)
    sv('figure_4_trust.png')
    plt.figure(figsize=(14,8))
    for g, vals in tg.items():
        plt.plot(steps, vals, alpha=0.75, label=g)
    plt.xlabel('Simulation Time Step (t)')
    plt.ylabel('True Positive Rate')
    plt.legend(bbox_to_anchor=(1.04,1), loc='upper left', fontsize=7)
    plt.grid(alpha=0.3)
    sv('figure_11_subgroup_tpr.png')
    daf_df = pd.DataFrame(dh).sort_index(axis=1)
    fig, ax = plt.subplots(figsize=(14,8))
    daf_df.plot(kind='area', stacked=True, colormap='viridis',
                alpha=0.6, ax=ax)
    ax.set_xlabel('Simulation Time Step (t)')
    ax.set_ylabel('Sample Weight (w_g,t)')
    ax.legend(bbox_to_anchor=(1.04,1), loc='upper left', fontsize=7)
    ax.grid(alpha=0.3)
    sv('figure_12_daf_weights.png')
    fig, ax = plt.subplots(figsize=(9,6))
    ax.boxplot([agg[m]['tpr_gap'] for m in
                ['Baseline','Static FAR','Dynamic DAF']],
               labels=['Baseline','Static FAR','Dynamic DAF'],
               patch_artist=True,
               boxprops=dict(facecolor='lightblue'),
               medianprops=dict(color='red', linewidth=2))
    ax.set_ylabel('Final TPR Gap')
    ax.grid(alpha=0.3)
    sv('figure_boxplot.png')


def _robustness_one(df, seed, shift=8, daf_lr=0.1, normalize_daf=True):
    T0=0.9; TF=0.2; ALPHA=0.025; BETA=0.01; GAMMA=0.005;
    np.random.seed(seed)
    drop = ['mortality_in_hospital','ethnicity','age_group',
             'intersectional_group','stay_id']
    pc = [c for c in df.columns if c not in drop]
    nf = df[pc].select_dtypes(include=np.number).columns.tolist()
    cf = df[pc].select_dtypes(include=['object','category']).columns.tolist()
    groups = sorted(df['intersectional_group'].unique())
    majority = df['intersectional_group'].value_counts().idxmax()
    model_names = ['Baseline','Static FAR','Dynamic DAF']
    hist = {m:{'tpr_gap':[],'fpr_gap':[]} for m in model_names}
    pools = {m: df.copy() for m in model_names}
    trust = T0
    daf_w = {g:1.0 for g in groups}
    shifted = False

    for step in range(N_STEPS):
        if step==shift and not shifted:
            rng = np.random.default_rng(seed+1)
            sg = rng.choice(groups, size=len(groups)//2, replace=False)
            for name in pools:
                pool = pools[name]
                mask = pool['intersectional_group'].isin(sg)
                if 'Lactate' in pool.columns:
                    pool.loc[mask,'Lactate'] *= 1.30
                if 'SOFA_score' in pool.columns:
                    pool.loc[mask,'SOFA_score'] = (
                        pool.loc[mask,'SOFA_score']+2).clip(upper=24)
            shifted = True

        last_daf = {}
        models_step = {}
        for name in model_names:
            pool = pools[name]
            tr, te = train_test_split(pool, test_size=0.3,
                                       random_state=seed+step,
                                       stratify=pool['mortality_in_hospital'])
            ytr, yte = tr['mortality_in_hospital'], te['mortality_in_hospital']
            gtr, gte = tr['intersectional_group'],  te['intersectional_group']
            Xtr, Xte = tr[pc], te[pc]
            pp = ColumnTransformer([
                ('num', StandardScaler(), nf),
                ('cat', OneHotEncoder(handle_unknown='ignore',
                                       sparse_output=False), cf)
            ], remainder='drop')
            if name=='Baseline':
                w = None
            elif name=='Static FAR':
                w = gtr.apply(lambda g: 1.0 if g==majority else 1.5).values
            else:
                w = gtr.map(daf_w).fillna(1.0).values
            model = Pipeline([('pp',pp),
                              ('clf',RandomForestClassifier(
                                  random_state=seed, class_weight='balanced',
                                  n_jobs=-1))])
            if w is None: model.fit(Xtr, ytr)
            else: model.fit(Xtr, ytr, clf__sample_weight=w)
            models_step[name] = (model, Xte, yte, gte)
            pr = model.predict(Xte)
            dfm, gaps = fairness_metrics(
                yte.reset_index(drop=True),
                pd.Series(pr),
                gte.reset_index(drop=True))
            hist[name]['tpr_gap'].append(gaps['tpr_gap'])
            hist[name]['fpr_gap'].append(gaps['fpr_gap'])
            if name=='Dynamic DAF': last_daf = dfm['TPR'].to_dict()

        if last_daf:
            ref = np.mean(list(last_daf.values()))
            for g in groups:
                daf_w[g] *= np.exp(daf_lr*(ref-last_daf.get(g,ref)))
            if normalize_daf:
                mean_w = np.mean(list(daf_w.values()))
                if mean_w > 0:
                    for g in groups: daf_w[g] /= mean_w

        for name in model_names:
            model, Xte, yte, gte = models_step[name]
            pool = pools[name]
            pr_b = model.predict(Xte)
            fp = int(((pr_b==1)&(yte==0)).sum())
            tp = int(((pr_b==1)&(yte==1)).sum())
            if name=='Baseline':
                trust += BETA*tp - ALPHA*fp - GAMMA*(fp+tp)/max(len(yte),1)
                trust = max(TF, min(1.0, trust))
            low = pool[model.predict(pool[pc])==0].index
            miss = low[np.random.rand(len(low)) < (1-trust)*0.5]
            if len(miss):
                for c in ['Heart Rate','Respiratory Rate']:
                    if c in pool.columns:
                        pool.loc[miss,c] *= (1+np.random.uniform(0.02,0.05,len(miss)))
                for c in ['Non Invasive Blood Pressure systolic',
                          'O2 saturation pulseoxymetry']:
                    if c in pool.columns:
                        pool.loc[miss,c] *= (1-np.random.uniform(0.02,0.05,len(miss)))
                if 'Lactate' in pool.columns:
                    pool.loc[miss,'Lactate'] *= (1+np.random.uniform(0.05,0.15,len(miss)))
    return hist


def robustness_experiment(df, daf_lr=0.1, normalize_daf=True):
    print('\n--- Robustness Experiment (30-run replicated) ---')
    SHIFT=8
    model_names = ['Baseline','Static FAR','Dynamic DAF']
    all_hist = {m:[] for m in model_names}
    for run in range(N_RUNS):
        seed = 999 + run*100
        hist = _robustness_one(df.copy(), seed, shift=SHIFT,
                               daf_lr=daf_lr, normalize_daf=normalize_daf)
        for m in model_names:
            all_hist[m].append(hist[m]['tpr_gap'])
        if (run+1)%5==0: print(f'  robustness {run+1}/{N_RUNS}')
    rows = []
    delta_arrays = {}
    for name in model_names:
        arr = np.asarray(all_hist[name], dtype=float)
        pre = arr[:,SHIFT-1]
        post = arr[:,SHIFT]
        delta = post - pre
        final = arr[:,-1]
        delta_arrays[name] = delta
        rows.append({'Model':name,
                     'TPR_pre':round(float(np.mean(pre)),4),
                     'TPR_pre_SD':round(float(np.std(pre)),4),
                     'TPR_post':round(float(np.mean(post)),4),
                     'TPR_post_SD':round(float(np.std(post)),4),
                     'Delta':round(float(np.mean(delta)),4),
                     'Delta_SD':round(float(np.std(delta)),4),
                     'TPR_final':round(float(np.mean(final)),4),
                     'TPR_final_SD':round(float(np.std(final)),4)})
    rob = pd.DataFrame(rows)
    rob.to_csv(os.path.join(OUT,'table_robustness.csv'), index=False)
    _paired_stats(delta_arrays, 'stats_robustness_delta.csv')

    # One-sided shock-attenuation Wilcoxon
    shock_rows = []
    for treatment in ['Static FAR','Dynamic DAF']:
        stat, p_two = wilcoxon(delta_arrays[treatment], delta_arrays['Baseline'])
        smaller = np.mean(delta_arrays[treatment]) < np.mean(delta_arrays['Baseline'])
        p_one = p_two/2 if smaller else 1 - p_two/2
        shock_rows.append({
            'Comparison': f'{treatment} vs Baseline shock magnitude',
            'Mean_Delta_treatment': round(float(np.mean(delta_arrays[treatment])),4),
            'Mean_Delta_baseline':  round(float(np.mean(delta_arrays['Baseline'])),4),
            'Difference':           round(float(np.mean(delta_arrays[treatment])-np.mean(delta_arrays['Baseline'])),4),
            'Wilcoxon_W':           round(float(stat),4),
            'p_two_sided':          round(float(p_two),6),
            'p_one_sided_smaller':  round(float(p_one),6)
        })
    pd.DataFrame(shock_rows).to_csv(os.path.join(OUT,'stats_shock_attenuation.csv'), index=False)
    print('\nShock-attenuation Wilcoxon:')
    for r in shock_rows:
        print(f"  {r['Comparison']}: p_one_sided={r['p_one_sided_smaller']}")

    st = {'Baseline':{'marker':'o','linestyle':'--','color':'steelblue'},
          'Static FAR':{'marker':'s','linestyle':':','color':'darkorange'},
          'Dynamic DAF':{'marker':'^','linestyle':'-','color':'green'}}
    plt.figure(figsize=(11,6))
    steps = np.arange(N_STEPS)
    for name, style in st.items():
        arr = np.asarray(all_hist[name], dtype=float)
        mean = arr.mean(axis=0)
        sd = arr.std(axis=0)
        plt.plot(steps, mean, **style, label=name)
        plt.fill_between(steps, mean-sd, mean+sd, alpha=0.15)
    plt.axvline(x=SHIFT, color='red', linestyle='--', linewidth=1.5,
                label=f'Exogenous shift (t={SHIFT})')
    plt.xlabel('Simulation Time Step (t)')
    plt.ylabel('Max-Min TPR Gap')
    plt.legend(); plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(OUT,'figure_10_robustness.png'), dpi=300)
    plt.close()
    print(rob.to_string(index=False))
    return rob


def eicu_fairness(df_mimic):
    print('\n--- eICU Cross-Institutional Fairness ---')
    try:
        pat = pd.read_csv(os.path.join(EICU,'patient.csv'),
                          usecols=['patientunitstayid','gender','age',
                                   'hospitalid','unitdischargestatus','ethnicity'])
    except Exception as e:
        print(f'eICU unavailable: {e}'); return None
    pat['age_num'] = pd.to_numeric(
        pat['age'].astype(str).str.replace('>',''), errors='coerce')
    pat = pat.dropna(subset=['age_num'])
    pat = pat[pat['age_num']>18]
    def mape(e):
        e = str(e).upper()
        if 'CAUCASIAN' in e: return 'White'
        if 'AFRICAN' in e: return 'Black'
        if 'ASIAN' in e or 'HISPANIC' in e: return 'Asian-Hispanic'
        return 'Other'
    pat['ethnicity'] = pat['ethnicity'].apply(mape)
    pat['age_group'] = np.where(pat['age_num']<65,'Under65','Over65')
    pat['gender_s'] = pat['gender'].str.upper().map(
        {'MALE':'M','FEMALE':'F'}).fillna('F')
    pat['intersectional_group'] = (
        pat['ethnicity']+'_'+pat['gender_s']+'_'+pat['age_group'])
    pat['mortality'] = (
        pat['unitdischargestatus'].str.upper()=='EXPIRED').astype(int)
    pat.rename(columns={'patientunitstayid':'stay_id',
                        'age_num':'anchor_age'}, inplace=True)

    try:
        vit_chunks = pd.read_csv(
            os.path.join(EICU,'vitalPeriodic.csv/vitalPeriodic.csv'),
            usecols=['patientunitstayid','heartrate','systemicsystolic',
                     'respiration','sao2','temperature'],
            iterator=True, chunksize=2_000_000)
        vit_df = pd.concat(vit_chunks)
        vit_df.rename(columns={'patientunitstayid':'stay_id',
                                'heartrate':'Heart Rate',
                                'systemicsystolic':'Non Invasive Blood Pressure systolic',
                                'respiration':'Respiratory Rate',
                                'sao2':'O2 saturation pulseoxymetry',
                                'temperature':'Temperature Fahrenheit'},
                       inplace=True)
        vit_agg = vit_df.groupby('stay_id')[VITAL_NAMES].mean().reset_index()
        pat = pat.merge(vit_agg, on='stay_id', how='left')
    except Exception as e:
        print(f'  vitals unavailable: {e}')
        for v in VITAL_NAMES:
            pat[v] = np.nan

    try:
        lab = pd.read_csv(os.path.join(EICU,'lab.csv'),
                          usecols=['patientunitstayid','labresultoffset',
                                   'labname','labresult'])
        lab.rename(columns={'patientunitstayid':'stay_id'}, inplace=True)
        lab = lab[lab['labresultoffset'].between(0,1440)]
        lm = {'creatinine':'Creatinine','wbc':'White Blood Cells',
              'potassium':'Potassium','bicarbonate':'Bicarbonate',
              'hgb':'Hemoglobin','glucose':'Glucose','lactate':'Lactate'}
        lab['labname'] = lab['labname'].str.lower()
        lab_p = (lab[lab['labname'].isin(lm)]
                 .assign(feat=lambda d: d['labname'].map(lm))
                 .groupby(['stay_id','feat'])['labresult'].mean()
                 .reset_index()
                 .pivot(index='stay_id', columns='feat', values='labresult')
                 .reset_index())
        pat = pat.merge(lab_p, on='stay_id', how='left')
    except Exception as e:
        print(f'  labs unavailable: {e}')

    common_features = VITAL_NAMES + list(LAB_IDS.values()) + ['anchor_age']
    for c in common_features:
        if c not in pat.columns:
            pat[c] = 0.0

    X_eicu = pat[common_features].fillna(0.0).astype(np.float32)
    y_eicu = pat['mortality'].values
    g_eicu = pat['intersectional_group'].values

    X_mimic = df_mimic[common_features].fillna(0.0).astype(np.float32)
    y_mimic = df_mimic['mortality_in_hospital'].values
    sc = StandardScaler()
    Xm = sc.fit_transform(X_mimic)
    Xe = sc.transform(X_eicu)
    rf = RandomForestClassifier(n_estimators=200, class_weight='balanced',
                                 random_state=42, n_jobs=-1)
    rf.fit(Xm, y_mimic)
    pb_e = rf.predict_proba(Xe)[:,1]
    pr_e = (pb_e>0.5).astype(int)
    auc_e, ci_lo, ci_hi = bootstrap_auc(y_eicu, pb_e)
    pd.DataFrame({'AUC':[auc_e],'CI_lo':[ci_lo],'CI_hi':[ci_hi]}).to_csv(
        os.path.join(OUT,'eicu_rf_auc.csv'), index=False)
    print(f'eICU RF AUC={auc_e} (95% CI: {ci_lo}-{ci_hi})')

    dfm_e, gaps_e = fairness_metrics(y_eicu, pr_e, g_eicu)
    dfm_e.round(4).to_csv(
        os.path.join(OUT,'table_eicu_fairness_overall.csv'))
    print(f'eICU TPR gap={gaps_e["tpr_gap"]:.4f} '
          f'FPR gap={gaps_e["fpr_gap"]:.4f} '
          f'FOR gap={gaps_e["for_gap"]:.4f}')

    hosp = []
    pat_with_pred = pat.copy()
    pat_with_pred['_pred'] = pr_e
    for hid, grp in pat_with_pred.groupby('hospitalid'):
        if len(grp) < 30: continue
        _, gps = fairness_metrics(
            grp['mortality'].values,
            grp['_pred'].values,
            grp['intersectional_group'].values)
        hosp.append({'hospitalid':hid,'n':len(grp),
                     'tpr_gap':round(gps['tpr_gap'],4),
                     'fpr_gap':round(gps['fpr_gap'],4),
                     'for_gap':round(gps['for_gap'],4)})
    n_hospitals = 0
    if hosp:
        hdf = pd.DataFrame(hosp).set_index('hospitalid').sort_values(
            'tpr_gap', ascending=False)
        hdf.to_csv(os.path.join(OUT,'table_eicu_fairness_hospital.csv'))
        fig, axes = plt.subplots(1,3, figsize=(14,5))
        for ax, col, lab in zip(axes,
                                  ['tpr_gap','fpr_gap','for_gap'],
                                  ['TPR Gap','FPR Gap','FOR Gap']):
            if col in hdf.columns and hdf[col].notna().any():
                ax.boxplot(hdf[col].dropna(), labels=[lab])
                ax.set_title(f'Cross-Institutional {lab}')
                ax.set_ylabel(f'Max-Min {lab}')
                ax.grid(alpha=0.3)
        fig.tight_layout()
        fig.savefig(os.path.join(OUT,'figure_eicu_hosp_fairness.png'),
                    dpi=300)
        plt.close()
        n_hospitals = len(hdf)
        print(f'Per-hospital fairness: {n_hospitals} hospitals analyzed')
    return {'rf_auc':auc_e,'ci_lo':ci_lo,'ci_hi':ci_hi,
            'tpr_gap':gaps_e['tpr_gap'],'fpr_gap':gaps_e['fpr_gap'],
            'for_gap':gaps_e['for_gap'], 'n_hospitals':n_hospitals}


def static_weight_static_audit_sensitivity(Xtr, ytr, gtr, Xte, yte, gte, nf, cf,
                                           weights=(1.0,1.1,1.2,1.3,1.5)):
    print('\n--- Static FAR Initial Weight Sensitivity ---')
    majority = gtr.value_counts().idxmax()
    rows = []
    for w in weights:
        pp = ColumnTransformer([
            ('num', StandardScaler(), nf),
            ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cf)
        ], remainder='drop')
        sample_w = None if w==1.0 else gtr.apply(
            lambda g: 1.0 if g==majority else w).values
        model = Pipeline([('pp',pp),
                          ('clf',RandomForestClassifier(
                              random_state=42, class_weight='balanced',
                              n_jobs=-1))])
        if sample_w is None:
            model.fit(Xtr, ytr)
        else:
            model.fit(Xtr, ytr, clf__sample_weight=sample_w)
        pr = model.predict(Xte)
        pb = model.predict_proba(Xte)[:,1]
        _, gaps = fairness_metrics(yte.reset_index(drop=True),
                                   pd.Series(pr),
                                   gte.reset_index(drop=True))
        rows.append({'Weight':w,
                     'AUC':round(float(roc_auc_score(yte,pb)),4),
                     'TPR_Gap':round(gaps['tpr_gap'],4),
                     'FPR_Gap':round(gaps['fpr_gap'],4),
                     'FOR_Gap':round(gaps['for_gap'],4),
                     'sigma_TPR':round(gaps['tpr_std'],4)})
    out = pd.DataFrame(rows)
    out.to_csv(os.path.join(OUT,'table_static_weight_initial_sensitivity.csv'),
               index=False)
    print(out.to_string(index=False))
    return out


def static_weight_longitudinal_sensitivity(df, weights=(1.1,1.2,1.3,1.5)):
    print('\n--- Static FAR Longitudinal Weight Sensitivity ---')
    rows = []
    for w in weights:
        vals = {'Final_TPR_Gap':[], 'Final_FPR_Gap':[], 'Final_FOR_Gap':[],
                'Mean_Abs_Cycle_Change':[], 'Max_Cycle_Jump':[],
                'Trajectory_Range':[]}
        for run in range(N_RUNS):
            seed = 2042 + int(round(w*100))*1000 + run*100
            rs, _, _, _ = run_one_sim(df.copy(), seed,
                                      capture_shap=False,
                                      static_weight=w,
                                      gamma=0.005)
            vals['Final_TPR_Gap'].append(rs['Static FAR']['final_tpr_gap'])
            vals['Final_FPR_Gap'].append(rs['Static FAR']['final_fpr_gap'])
            vals['Final_FOR_Gap'].append(rs['Static FAR']['final_for_gap'])
            met = _trajectory_metrics(rs['Static FAR']['tpr_gap_series'])
            for k in ['Mean_Abs_Cycle_Change','Max_Cycle_Jump','Trajectory_Range']:
                vals[k].append(met[k])
        row = {'Weight':w}
        for k, arr in vals.items():
            arr = np.asarray(arr, dtype=float)
            row[f'{k}_mean'] = round(float(np.mean(arr)),4)
            row[f'{k}_SD'] = round(float(np.std(arr)),4)
        rows.append(row)
        print(f'  static weight {w}: done')
    out = pd.DataFrame(rows)
    out.to_csv(os.path.join(OUT,'table_static_weight_longitudinal_sensitivity.csv'),
               index=False)
    print(out.to_string(index=False))
    return out


def gamma_sensitivity(df, gammas=(0.0,0.005,0.01,0.02)):
    print('\n--- Gamma Sensitivity ---')
    model_names = ['Baseline','Static FAR','Dynamic DAF']
    rows = []
    for gamma in gammas:
        vals = {m:{'Final_TPR_Gap':[], 'Final_FPR_Gap':[], 'Final_FOR_Gap':[],
                   'Mean_Abs_Cycle_Change':[]} for m in model_names}
        for run in range(N_RUNS):
            seed = 3042 + int(round(gamma*100000))*100 + run*100
            rs, _, _, _ = run_one_sim(df.copy(), seed,
                                      capture_shap=False,
                                      static_weight=1.5,
                                      gamma=gamma)
            for m in model_names:
                vals[m]['Final_TPR_Gap'].append(rs[m]['final_tpr_gap'])
                vals[m]['Final_FPR_Gap'].append(rs[m]['final_fpr_gap'])
                vals[m]['Final_FOR_Gap'].append(rs[m]['final_for_gap'])
                vals[m]['Mean_Abs_Cycle_Change'].append(
                    _trajectory_metrics(rs[m]['tpr_gap_series'])['Mean_Abs_Cycle_Change'])
        for m in model_names:
            row = {'Gamma':gamma, 'Model':m}
            for k, arr in vals[m].items():
                arr = np.asarray(arr, dtype=float)
                row[f'{k}_mean'] = round(float(np.mean(arr)),4)
                row[f'{k}_SD'] = round(float(np.std(arr)),4)
            rows.append(row)
        print(f'  gamma {gamma}: done')
    out = pd.DataFrame(rows)
    out.to_csv(os.path.join(OUT,'table_gamma_sensitivity.csv'), index=False)
    print(out.to_string(index=False))
    return out


def save_key_numbers(cs, ls, es, audit, sim, stat, rob, shap_r, env, extra_outputs=None):
    rows = []
    def a(loc, val, note=''):
        rows.append({'Location':loc,'Value':val,'Note':note})
    a('TableI.SOFA_mean', cs['SOFA_Mean'])
    a('TableI.SOFA_SD', cs['SOFA_SD'])
    a('TableI.Vasopressor_pct', cs['Vasopressor_pct'])
    a('TableI.MV_pct', cs['MV_pct'])
    a('LSTM.MIMIC_AUC', ls['auc'],
      f"95% CI: {ls['ci_lo']}-{ls['ci_hi']}")
    a('LSTM.CI_lo', ls['ci_lo'])
    a('LSTM.CI_hi', ls['ci_hi'])
    if es:
        a('eICU.RF_AUC', es['rf_auc'],
          f"95% CI: {es['ci_lo']}-{es['ci_hi']}")
        a('eICU.TPR_gap', es['tpr_gap'])
        a('eICU.FPR_gap', es['fpr_gap'])
        a('eICU.FOR_gap', es['for_gap'])
        if 'n_hospitals' in es:
            a('eICU.n_hospitals_ge30', es['n_hospitals'])
    for m in audit.index:
        t = m.replace(' ','_')
        for c in audit.columns:
            a(f'TableIV.{t}.{c}', audit.loc[m,c])
    for m in sim.index:
        t = m.replace(' ','_')
        for c in sim.columns:
            a(f'TableVI.{t}.{c}', sim.loc[m,c])
    a('Stats.Friedman_TPR_chi2', stat['friedman_chi2'])
    a('Stats.Friedman_TPR_p', stat['friedman_p'])
    a('Stats.Friedman_FPR_chi2', stat['fpr_friedman_chi2'])
    a('Stats.Friedman_FPR_p', stat['fpr_friedman_p'])
    for r in stat['wilcoxon']:
        t = r['Comparison'].replace(' ','_')
        a(f'Stats.Wilcoxon.{t}.p', r['p'])
    if shap_r:
        a('SHAP.rho', shap_r.get('Spearman_rho','N/A'))
        a('SHAP.p', shap_r.get('p_value','N/A'))
    if rob is not None:
        for _, row in rob.iterrows():
            m = str(row['Model']).replace(' ','_')
            for c in rob.columns:
                if c == 'Model': continue
                a(f'Rob.{m}.{c}', row[c])
    if extra_outputs:
        for name, table in extra_outputs.items():
            if table is None: continue
            for _, row in table.iterrows():
                if 'Model' in row.index:
                    prefix = f'{name}.{str(row["Model"]).replace(" ","_")}'
                elif 'Weight' in row.index:
                    prefix = f'{name}.weight_{row["Weight"]}'
                elif 'Gamma' in row.index:
                    prefix = f'{name}.gamma_{row["Gamma"]}'
                else:
                    prefix = name
                for c in table.columns:
                    if c in ['Model','Weight','Gamma']: continue
                    a(f'{prefix}.{c}', row[c])
    for k, v in env.items():
        a(f'Env.{k}', v)
    df_out = pd.DataFrame(rows)
    df_out.to_csv(os.path.join(OUT,'manuscript_key_numbers.csv'),
                   index=False)
    with open(os.path.join(OUT,'manuscript_key_numbers.txt'), 'w') as f:
        f.write('='*60+'\nMANUSCRIPT KEY NUMBERS\n'+'='*60+'\n\n')
        for _, r in df_out.iterrows():
            f.write(f"{r['Location']:<50} {r['Value']}"
                    + (f"  [{r['Note']}]" if r['Note'] else '')
                    + '\n')


def main():
    env = log_env()
    proc = os.path.join(OUT,'sepsis_cohort_17feat.csv')
    need = ['Lactate','vasopressor_use','mechanical_ventilation','SOFA_score']
    if (os.path.exists(proc) and
        os.path.exists(os.path.join(OUT,'intime_map.csv'))):
        tmp = pd.read_csv(proc, nrows=1)
        if all(c in tmp.columns for c in need) and 'stay_id' in tmp.columns:
            print('Loading existing data.')
            df = pd.read_csv(proc)
            it_df = pd.read_csv(os.path.join(OUT,'intime_map.csv'))
            intime_map = dict(zip(it_df['stay_id'], it_df['intime']))
        else:
            df, intime_map = preprocess_mimic()
    else:
        df, intime_map = preprocess_mimic()

    cs = cohort_analysis(df)

    (lstm_m, st_obj, X_seq, vmeans, vstds, static_cols,
     num_cols, cat_cols, ls) = ml_baselines(df, intime_map)

    drop = ['mortality_in_hospital','ethnicity','age_group',
             'intersectional_group','stay_id']
    X_static = df[[c for c in df.columns if c not in drop+VITAL_NAMES]]
    y = df['mortality_in_hospital']
    g = df['intersectional_group']
    Xtr, Xte, ytr, yte, gtr, gte = train_test_split(
        X_static, y, g, test_size=0.2, random_state=42, stratify=y)
    nf = Xtr.select_dtypes(include=np.number).columns.tolist()
    cf = Xtr.select_dtypes(include=['object','category']).columns.tolist()

    majority = gtr.value_counts().idxmax()
    print(f'Static audit majority anchor: {majority}')

    pp0 = ColumnTransformer([
        ('num', StandardScaler(), nf),
        ('cat', OneHotEncoder(handle_unknown='ignore',
                               sparse_output=False), cf)
    ], remainder='drop')
    bm = Pipeline([('pp',pp0),
                    ('clf',RandomForestClassifier(
                        random_state=42, class_weight='balanced',
                        n_jobs=-1))]).fit(Xtr, ytr)
    sw = gtr.apply(lambda g: 1.0 if g==majority else 1.5).values
    sm = Pipeline([('pp',pp0),
                    ('clf',RandomForestClassifier(
                        random_state=42, class_weight='balanced',
                        n_jobs=-1))]).fit(Xtr, ytr,
                                          clf__sample_weight=sw)
    audit = static_audit({'Baseline':bm,'Static Reweighing':sm},
                          Xte, yte, gte)
    static_initial_sens = static_weight_static_audit_sensitivity(
        Xtr, ytr, gtr, Xte, yte, gte, nf, cf)

    # --- Standard simulation (default η=0.1, normalization on) ---
    sim, stat = run_simulation(df.copy(), daf_lr=0.1, normalize_daf=True)
    shap_r = shap_spearman()
    rob = robustness_experiment(df.copy(), daf_lr=0.1, normalize_daf=True)

    # --- Optional: Run sensitivity experiments (uncomment to run) ---
    # # DAF without normalization
    # sim_no_norm, stat_no_norm = run_simulation(df.copy(), daf_lr=0.1, normalize_daf=False)
    # rob_no_norm = robustness_experiment(df.copy(), daf_lr=0.1, normalize_daf=False)
    # # DAF with larger η = 0.2
    # sim_eta02, stat_eta02 = run_simulation(df.copy(), daf_lr=0.2, normalize_daf=True)
    # rob_eta02 = robustness_experiment(df.copy(), daf_lr=0.2, normalize_daf=True)
    # # DAF with larger η = 0.3
    # sim_eta03, stat_eta03 = run_simulation(df.copy(), daf_lr=0.3, normalize_daf=True)
    # rob_eta03 = robustness_experiment(df.copy(), daf_lr=0.3, normalize_daf=True)

    static_long_sens = static_weight_longitudinal_sensitivity(df.copy())
    gamma_sens = gamma_sensitivity(df.copy())
    es = eicu_fairness(df)

    extra_outputs = {
        'StaticInitialSensitivity': static_initial_sens,
        'StaticLongitudinalSensitivity': static_long_sens,
        'GammaSensitivity': gamma_sens
    }
    save_key_numbers(cs, ls, es or {}, audit, sim, stat, rob, shap_r, env,
                     extra_outputs=extra_outputs)
    try:
        if '__file__' in globals():
            shutil.copyfile(__file__, os.path.join(OUT, 'pipeline_code.py'))
    except Exception:
        pass
    shutil.make_archive('/kaggle/working/output_bundle', 'zip', OUT)
    print('\nDone. Download output_bundle.zip')


if __name__=='__main__':
    main()